In [1]:
!pip install snntorch -q

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import snntorch as snn

from snntorch import surrogate

from torch.utils.data import Dataset, DataLoader

PT_DATASET = "/content/drive/MyDrive/DVS128_PT"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.8 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [3]:
NUM_CLASSES = 11

beta = 0.95

spike_grad = surrogate.fast_sigmoid()

In [4]:
class DVSGestureSNN(nn.Module):

    def __init__(self):

        super().__init__()

        # ---------- Feature Extraction ----------

        self.conv1 = nn.Conv2d(
            2,
            16,
            kernel_size=3,
            padding=1
        )

        self.pool1 = nn.MaxPool2d(2)

        self.lif1 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad
        )

        self.conv2 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.pool2 = nn.MaxPool2d(2)

        self.lif2 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad
        )

        self.conv3 = nn.Conv2d(
            32,
            64,
            kernel_size=3,
            padding=1
        )

        self.pool3 = nn.MaxPool2d(2)

        self.lif3 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad
        )

        # ---------- Classifier ----------

        self.fc = nn.Linear(
            64 * 16 * 16,
            NUM_CLASSES
        )

    def forward(self, x):

        """
        x shape:
        (batch, 30, 2, 128, 128)
        """

        batch_size = x.size(0)

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        spike_sum = torch.zeros(
            batch_size,
            NUM_CLASSES,
            device=x.device
        )

        # Process each time bin
        for t in range(x.size(1)):

            out = x[:, t]

            out = self.conv1(out)
            out = self.pool1(out)
            spk1, mem1 = self.lif1(out, mem1)

            out = self.conv2(spk1)
            out = self.pool2(out)
            spk2, mem2 = self.lif2(out, mem2)

            out = self.conv3(spk2)
            out = self.pool3(out)
            spk3, mem3 = self.lif3(out, mem3)

            out = out.reshape(batch_size, -1)

            out = self.fc(out)

            spike_sum += out

        return spike_sum / x.size(1)

In [5]:
import pandas as pd

metadata = pd.read_csv(
    os.path.join(PT_DATASET, "metadata.csv")
)

print(metadata.head())
print()
print("Total samples:", len(metadata))

train_metadata = metadata[
    metadata["split"] == "train"
].reset_index(drop=True)

test_metadata = metadata[
    metadata["split"] == "test"
].reset_index(drop=True)

print(f"Train samples: {len(train_metadata)}")
print(f"Test samples : {len(test_metadata)}")

              file  label  split
0  sample_00000.pt      0  train
1  sample_00001.pt      1  train
2  sample_00002.pt      2  train
3  sample_00003.pt      3  train
4  sample_00004.pt      4  train

Total samples: 1464
Train samples: 1176
Test samples : 288


In [6]:
class PTGestureDataset(Dataset):

    def __init__(self, metadata):

        self.metadata = metadata

    def __len__(self):

        return len(self.metadata)

    def __getitem__(self, index):

        sample = self.metadata.iloc[index]

        voxel = torch.load(
            os.path.join(
                PT_DATASET,
                sample["file"]
            )
        )

        # Convert int8 back to float32
        voxel = voxel.float() / 127.0

        label = torch.tensor(
            sample["label"],
            dtype=torch.long
        )

        return voxel, label

In [7]:
train_dataset = PTGestureDataset(train_metadata)

test_dataset = PTGestureDataset(test_metadata)

print(len(train_dataset))
print(len(test_dataset))

1176
288


In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [9]:
images, labels = next(iter(train_loader))

print(images.dtype)
print(images.min())
print(images.max())
print(images.mean())

print(torch.unique(images))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.float32
tensor(0.)
tensor(1.)
tensor(0.0056)
tensor([0.0000, 0.0079, 0.0157, 0.0236, 0.0315, 0.0394, 0.0472, 0.0551, 0.0630,
        0.0709, 0.0787, 0.0866, 0.0945, 0.1024, 0.1102, 0.1181, 0.1260, 0.1339,
        0.1417, 0.1496, 0.1575, 0.1654, 0.1732, 0.1811, 0.1890, 0.1969, 0.2047,
        0.2126, 0.2205, 0.2283, 0.2362, 0.2441, 0.2520, 0.2598, 0.2677, 0.2756,
        0.2835, 0.2913, 0.2992, 0.3071, 0.3150, 0.3228, 0.3307, 0.3386, 0.3465,
        0.3543, 0.3622, 0.3701, 0.3780, 0.3858, 0.3937, 0.4016, 0.4094, 0.4173,
        0.4252, 0.4331, 0.4409, 0.4488, 0.4567, 0.4646, 0.4724, 0.4803, 0.4882,
        0.4961, 0.5039, 0.5118, 0.5197, 0.5276, 0.5354, 0.5433, 0.5512, 0.5591,
        0.5669, 0.5748, 0.5827, 0.5906, 0.5984, 0.6063, 0.6142, 0.6220, 0.6299,
        0.6378, 0.6457, 0.6535, 0.6614, 0.6693, 0.6772, 0.6850, 0.6929, 0.7008,
        0.7087, 0.7165, 0.7244, 0.7323, 0.7402, 0.7480, 0.7559, 0.7638, 0.7717,
        0.7795, 0.7953, 0.8031, 0.8189, 0.8268, 0.8425, 0.8504, 0.866

In [10]:
model = DVSGestureSNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Model ready!")

Model ready!


In [11]:
print("Device:", device)

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Output shape:", outputs.shape)

Device: cpu


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Output shape: torch.Size([16, 11])


In [12]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predicted = outputs.argmax(1)

        correct += (predicted == labels).sum().item()

        total += labels.size(0)

    loss = running_loss / len(loader)
    accuracy = 100 * correct / total

    return loss, accuracy

In [13]:
def evaluate(model, loader):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predicted = outputs.argmax(1)

            correct += (predicted == labels).sum().item()

            total += labels.size(0)

    loss = running_loss / len(loader)
    accuracy = 100 * correct / total

    return loss, accuracy

In [14]:
EPOCHS = 5

train_loss_history = []
test_loss_history = []

train_acc_history = []
test_acc_history = []

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader
    )

    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)

    train_acc_history.append(train_acc)
    test_acc_history.append(test_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" | Train Loss {train_loss:.4f}"
        f" | Train Acc {train_acc:.2f}%"
        f" | Test Loss {test_loss:.4f}"
        f" | Test Acc {test_acc:.2f}%"
    )

Epoch 1/5 | Train Loss 1.5351 | Train Acc 43.54% | Test Loss 0.9816 | Test Acc 59.03%
Epoch 2/5 | Train Loss 0.8459 | Train Acc 65.90% | Test Loss 0.9015 | Test Acc 67.01%
Epoch 3/5 | Train Loss 0.6999 | Train Acc 71.43% | Test Loss 0.9808 | Test Acc 63.19%
Epoch 4/5 | Train Loss 0.7030 | Train Acc 71.26% | Test Loss 1.0199 | Test Acc 66.67%
Epoch 5/5 | Train Loss 0.5927 | Train Acc 75.09% | Test Loss 1.0504 | Test Acc 67.36%


In [15]:
MODEL_PATH = "/content/drive/MyDrive/DVS128_PT/dvs_snn_model.pth"

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved to:", MODEL_PATH)

Model saved to: /content/drive/MyDrive/DVS128_PT/dvs_snn_model.pth
